# 05 — Retrieval & Similarity Search (Top-k, Scores & MMR Diversity)

**By the end of this notebook, you can inspect similarity scores, filter irrelevant hits, and use MMR retrieval to return diverse context without duplicate chunks.**

### Learning Objectives
- Use `similarity_search_with_score` to inspect confidence values and reject low-relevance garbage.
- Compare plain similarity search against Maximal Marginal Relevance (`search_type='mmr'`) for multi-topic diversity.
- Apply retrieval-time metadata filters to restrict search by source file or department.

### Prerequisites
- Notebook 04 (`04_embeddings_and_vector_stores.ipynb`)

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: Similarity Search vs. MMR Diversity

Plain **Top-k Similarity Search** pulls the *k* chunks closest to your question — which often means grabbing 3 slightly shifted overlapping windows from the exact same paragraph!

**Maximal Marginal Relevance (MMR)** solves redundancy like hiring a diverse project team:
1. Fetch a broader pool (`fetch_k=10` candidates).
2. Pick the single best candidate first.
3. For each remaining slot up to `k`, reward chunks that are **relevant to the query** YET **different from chunks already selected** (`lambda_mult` balances relevance vs diversity).

```
Standard Top-3:  [ Chunk 1 (handbook sec 4), Chunk 2 (handbook sec 4 overlap), Chunk 3 (handbook sec 4) ]
MMR (Top-3)   :  [ Chunk 1 (handbook sec 4), Chunk A (product_faq ZeroGravity), Chunk X (changelog note) ]
```

## 3. Step-by-Step Build

### Step 1: Index the Full Novasphere Corpus
Let's load all four Novasphere files (including `policy_brief.pdf`), split cleanly, and populate an `InMemoryVectorStore`.

In [2]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=340, chunk_overlap=45).split_documents(load_corpus_documents(data_dir))
store = InMemoryVectorStore.from_documents(chunks, embeddings)
print(f"Indexed {len(chunks)} chunks across handbook, FAQ, changelog, and PDF brief.")
        

Indexed 33 chunks across handbook, FAQ, changelog, and PDF brief.


> **What just happened?**
> All documents are now indexed together with `source` and `doc_type` metadata preserved.

### Step 2: Inspect Raw Similarity Scores (Relevant vs. Off-Topic Query)
Let's run `similarity_search_with_score` on an on-topic query (`Enterprise Plan cost`) vs an off-topic query (`banana smoothie recipe`) to see how confidence shifts.

In [3]:
for q in ["How much does the Enterprise Plan cost per month?", "How do you blend a banana strawberry smoothie?"]:
    scored = store.similarity_search_with_score(q, k=2)
    top_doc, top_score = scored[0]
    print(f"Query: {q[:44]:<44} | top_score={top_score:.3f} | src={top_doc.metadata['source']}")
        

Query: How much does the Enterprise Plan cost per m | top_score=0.669 | src=company_handbook.txt
Query: How do you blend a banana strawberry smoothi | top_score=0.062 | src=company_handbook.txt


> **What just happened?**
> On-topic questions score substantially higher than off-topic noise—giving us a threshold signal!

### Step 3: Compare Standard Similarity Search vs. MMR Diversity
When a broad question touches multiple tiers or files, watch how MMR pulls a wider variety of document sources than plain Top-k.

In [4]:
broad_q = "What plans include ZeroGravity failover, HyperSync telemetry, and ROS2 support?"
sim_sources = [d.metadata["source"] for d in store.similarity_search(broad_q, k=4)]
mmr_sources = [d.metadata["source"] for d in store.max_marginal_relevance_search(broad_q, k=4, fetch_k=12, lambda_mult=0.5)]
print("Plain Top-4 sources:", sim_sources)
print("MMR Top-4 sources  :", mmr_sources)
        

Plain Top-4 sources: ['product_faq.md', 'product_faq.md', 'company_handbook.txt', 'company_handbook.txt']
MMR Top-4 sources  : ['product_faq.md', 'product_faq.md', 'recent_changelog.md', 'company_handbook.txt']


> **What just happened?**
> MMR reduces repetition and spreads coverage across `company_handbook.txt`, `product_faq.md`, and `recent_changelog.md`.

### Step 4: Retrieve with Clean Metadata Filters
Sometimes users only want recent updates (`recent_changelog.md`) or HR rules (`company_handbook.txt`). Passing a `filter` function narrows search instantly.

In [5]:
faq_only = store.similarity_search(
    "What is the failover threshold?",
    k=2,
    filter=lambda doc: doc.metadata.get("source") == "product_faq.md",
)
for d in faq_only:
    print(f"[{d.metadata['source']}] {d.page_content[:110].replace(chr(10), ' ')}...")
        

[product_faq.md] ### How does ZeroGravity Failover work, and which plan includes it?...
[product_faq.md] ZeroGravity Automatic Failover is exclusively included in the **Enterprise Plan** ($4,200/month). If a primary...


> **What just happened?**
> Metadata filtering prevents old manuals or wrong departments from ever polluting the retrieved context.

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Tune `lambda_mult` in MMR** — *Hint:* Compare `lambda_mult=0.1` (extreme diversity) vs `lambda_mult=0.9` (near pure similarity).
2. **Implement a threshold helper** — *Hint:* Filter `similarity_search_with_score(q, k=4)` to keep only chunks scoring >= 0.25.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
q_test = "ROS2 distributions supported"
hits_with_scores = store.similarity_search_with_score(q_test, k=4)
filtered = [d for d, s in hits_with_scores if s >= 0.25]
print(f"Kept {len(filtered)}/{len(hits_with_scores)} hits above score threshold 0.25.")
        

Kept 4/4 hits above score threshold 0.25.


## 5. Common Mistakes
- Assuming higher `k` (like `k=20`) is always better — flooding prompts with marginally related text dilutes key facts.
- Ignoring score thresholds when users ask out-of-scope questions, forcing unrelated chunks into the LLM.
- Setting `fetch_k < k` in MMR — always make `fetch_k` 2x–4x larger than `k` so diversity has candidates to pick from.

## 6. Recap
- `similarity_search_with_score` lets you inspect retrieval confidence before generating an answer.
- MMR balances relevance and diversity so multi-part questions cover distinct source chunks.
- Metadata filters guarantee strict scoping by document type, date, or department.

## 7. Next Notebook
Next up (`06_generation_and_prompting.ipynb`): Wire retrievers to grounded prompts with LangChain LCEL (`|`) chains and teach the model to refuse when evidence is missing!

## 8. Video Chapter Outline
1. **Why Raw Top-k Repeats Itself** — The overlapping chunk problem explained.
2. **Similarity Scores & Confidence** — Separating real matches from off-topic queries.
3. **MMR Retrieval Deep Dive** — How `fetch_k` and `lambda_mult` diversify context.
4. **Pre-Filtering with Metadata** — Zero-noise scoping per file or doc type.
5. **Hands-On Lab & Recap** — Building a threshold-guarded retriever.